In [1]:
import os
import numpy as np
import pandas as pd
import librosa
from tqdm import tqdm

import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint

2026-04-10 18:10:20.068276: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1775844620.311101      23 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1775844620.381195      23 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1775844620.917811      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1775844620.917881      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1775844620.917885      23 computation_placer.cc:177] computation placer alr

In [2]:
BASE_PATH = "/kaggle/input/competitions/digitrecognition-ee708"

TRAIN_AUDIO_PATH = f"{BASE_PATH}/train_audio//train_audio"
TEST_AUDIO_PATH = f"{BASE_PATH}/test_audio//test_audio"

SR = 16000
DURATION = 1
SAMPLES = SR * DURATION

In [3]:
import os

files = os.listdir(TRAIN_AUDIO_PATH)
print(files[:10])

['train_013107.wav', 'train_017985.wav', 'train_037638.wav', 'train_035520.wav', 'train_020499.wav', 'train_033310.wav', 'train_029992.wav', 'train_005075.wav', 'train_001574.wav', 'train_005291.wav']


In [4]:
train_df = pd.read_csv(f"{BASE_PATH}/train.csv")

print(train_df.columns)
print(train_df.head())

Index(['id', 'label'], dtype='object')
             id  label
0  train_000001      6
1  train_000002      6
2  train_000003      7
3  train_000004      7
4  train_000005      4


In [5]:
def extract_features(file_path, n_mels):
    audio, _ = librosa.load(file_path, sr=SR)
    
    if len(audio) < SAMPLES:
        audio = np.pad(audio, (0, SAMPLES - len(audio)))
    else:
        audio = audio[:SAMPLES]
    
    mel = librosa.feature.melspectrogram(
        y=audio, sr=SR, n_mels=n_mels
    )
    mel_db = librosa.power_to_db(mel, ref=np.max)
    
    return mel_db

In [6]:
def load_data(n_mels):
    train_df = pd.read_csv(f"{BASE_PATH}/train.csv")
    
    X, y = [], []
    
    for _, row in tqdm(train_df.iterrows(), total=len(train_df)):
        file_path = f"{TRAIN_AUDIO_PATH}/{row['id']}.wav"
        mel = extract_features(file_path, n_mels)
        
        X.append(mel)
        y.append(row['label'])
    
    X = np.array(X)[..., np.newaxis]
    y = np.array(y)
    
    return X, y

In [7]:
def build_model(input_shape):
    model = models.Sequential([
        # Block 1
        layers.Conv2D(32, (3,3), padding='same', activation='relu', input_shape=input_shape),
        layers.BatchNormalization(),
        layers.Conv2D(32, (3,3), padding='same', activation='relu'),
        layers.BatchNormalization(),
        layers.MaxPooling2D(2,2),

        # Block 2
        layers.Conv2D(64, (3,3), padding='same', activation='relu'),
        layers.BatchNormalization(),
        layers.Conv2D(64, (3,3), padding='same', activation='relu'),
        layers.BatchNormalization(),
        layers.MaxPooling2D(2,2),

        # Block 3
        layers.Conv2D(128, (3,3), padding='same', activation='relu'),
        layers.BatchNormalization(),
        layers.MaxPooling2D(2,2),

        # Block 4 (new)
        layers.Conv2D(256, (3,3), padding='same', activation='relu'),
        layers.BatchNormalization(),
        layers.MaxPooling2D(2,2),

        # Head
        layers.GlobalAveragePooling2D(),
        layers.Dense(128, activation='relu'),
        layers.Dropout(0.5),
        layers.Dense(10, activation='softmax')
    ])

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.0003),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )

    return model

In [8]:
from tensorflow.keras.callbacks import ModelCheckpoint, ReduceLROnPlateau

checkpoint_64 = ModelCheckpoint(
    "best_model_64.h5",
    monitor='val_accuracy',
    save_best_only=True,
    mode='max',
    verbose=1
)

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.3,
    patience=3,
    min_lr=1e-6,
    verbose=1
)

In [9]:
X64, y64 = load_data(64)

model_64 = build_model(X64.shape[1:])

history_64 = model_64.fit(
    X64, y64,
    validation_split=0.2,
    epochs=40,   # increased epochs
    batch_size=32,
    callbacks=[checkpoint_64, reduce_lr]
)

100%|██████████| 37800/37800 [07:39<00:00, 82.31it/s]
/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
I0000 00:00:1775845108.078147      23 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1775845108.084018      23 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13757 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


Epoch 1/40


I0000 00:00:1775845113.741518      80 service.cc:152] XLA service 0x7df2e8008030 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1775845113.741552      80 service.cc:160]   StreamExecutor device (0): Tesla T4, Compute Capability 7.5
I0000 00:00:1775845113.741558      80 service.cc:160]   StreamExecutor device (1): Tesla T4, Compute Capability 7.5
I0000 00:00:1775845114.532932      80 cuda_dnn.cc:529] Loaded cuDNN version 91002
2026-04-10 18:18:36.824884: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-04-10 18:18:36.967188: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.


 16/945 ━━━━━━━━━━━━━━━━━━━━ 10s 11ms/step - accuracy: 0.1408 - loss: 2.6436

I0000 00:00:1775845120.462204      80 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


...

945/945 ━━━━━━━━━━━━━━━━━━━━ 8s 8ms/step - accuracy: 0.9895 - loss: 0.0334 - val_accuracy: 0.9872 - val_loss: 0.0377 - learning_rate: 3.0000e-04
Epoch 8/40
939/945 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.9912 - loss: 0.0262
Epoch 8: val_accuracy did not improve from 0.98717
945/945 ━━━━━━━━━━━━━━━━━━━━ 8s 8ms/step - accuracy: 0.9912 - loss: 0.0262 - val_accuracy: 0.9839 - val_loss: 0.0497 - learning_rate: 3.0000e-04
Epoch 9/40
945/945 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.9917 - loss: 0.0263
Epoch 9: val_accuracy did not improve from 0.98717
945/945 ━━━━━━━━━━━━━━━━━━━━ 8s 8ms/step - accuracy: 0.9917 - loss: 0.0263 - val_accuracy: 0.9845 - val_loss: 0.0461 - learning_rate: 3.0000e-04
Epoch 10/40
939/945 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.9926 - loss: 0.0237
Epoch 10: val_accuracy did not improve from 0.98717

Epoch 10: ReduceLROnPlateau reducing learning rate to 9.000000427477062e-05.
945/945 ━━━━━━━━━━━━━━━━━━━━ 8s 8ms/step - accuracy: 0.9926 - loss: 0.0237

945/945 ━━━━━━━━━━━━━━━━━━━━ 8s 8ms/step - accuracy: 0.9965 - loss: 0.0101 - val_accuracy: 0.9910 - val_loss: 0.0269 - learning_rate: 9.0000e-05
Epoch 12/40
945/945 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.9986 - loss: 0.0052
Epoch 12: val_accuracy improved from 0.99101 to 0.99233, saving model to best_model_64.h5


945/945 ━━━━━━━━━━━━━━━━━━━━ 8s 8ms/step - accuracy: 0.9986 - loss: 0.0052 - val_accuracy: 0.9923 - val_loss: 0.0238 - learning_rate: 9.0000e-05
Epoch 13/40
945/945 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.9993 - loss: 0.0035
Epoch 13: val_accuracy did not improve from 0.99233
945/945 ━━━━━━━━━━━━━━━━━━━━ 8s 8ms/step - accuracy: 0.9993 - loss: 0.0035 - val_accuracy: 0.9909 - val_loss: 0.0307 - learning_rate: 9.0000e-05
Epoch 14/40
939/945 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.9995 - loss: 0.0029
Epoch 14: val_accuracy did not improve from 0.99233
945/945 ━━━━━━━━━━━━━━━━━━━━ 8s 8ms/step - accuracy: 0.9995 - loss: 0.0029 - val_accuracy: 0.9914 - val_loss: 0.0298 - learning_rate: 9.0000e-05
Epoch 15/40
939/945 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.9990 - loss: 0.0045
Epoch 15: val_accuracy did not improve from 0.99233

Epoch 15: ReduceLROnPlateau reducing learning rate to 2.700000040931627e-05.
945/945 ━━━━━━━━━━━━━━━━━━━━ 8s 8ms/step - accuracy: 0.9990 - loss: 0.


Epoch 18: ReduceLROnPlateau reducing learning rate to 8.100000013655517e-06.
945/945 ━━━━━━━━━━━━━━━━━━━━ 8s 8ms/step - accuracy: 0.9999 - loss: 9.1229e-04 - val_accuracy: 0.9927 - val_loss: 0.0262 - learning_rate: 2.7000e-05
Epoch 19/40
939/945 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.9998 - loss: 0.0011
Epoch 19: val_accuracy did not improve from 0.99272
945/945 ━━━━━━━━━━━━━━━━━━━━ 8s 8ms/step - accuracy: 0.9998 - loss: 0.0011 - val_accuracy: 0.9926 - val_loss: 0.0251 - learning_rate: 8.1000e-06
Epoch 20/40
939/945 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.9999 - loss: 6.7304e-04
Epoch 20: val_accuracy did not improve from 0.99272
945/945 ━━━━━━━━━━━━━━━━━━━━ 8s 8ms/step - accuracy: 0.9999 - loss: 6.7394e-04 - val_accuracy: 0.9923 - val_loss: 0.0256 - learning_rate: 8.1000e-06
Epoch 21/40
945/945 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.9999 - loss: 6.9679e-04
Epoch 21: val_accuracy did not improve from 0.99272

Epoch 21: ReduceLROnPlateau reducing learning rate to

945/945 ━━━━━━━━━━━━━━━━━━━━ 8s 8ms/step - accuracy: 1.0000 - loss: 5.5727e-04 - val_accuracy: 0.9929 - val_loss: 0.0259 - learning_rate: 1.0000e-06
Epoch 37/40
939/945 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 1.0000 - loss: 5.8595e-04
Epoch 37: val_accuracy did not improve from 0.99286
945/945 ━━━━━━━━━━━━━━━━━━━━ 8s 8ms/step - accuracy: 1.0000 - loss: 5.8642e-04 - val_accuracy: 0.9925 - val_loss: 0.0260 - learning_rate: 1.0000e-06
Epoch 38/40
945/945 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 1.0000 - loss: 4.6432e-04
Epoch 38: val_accuracy did not improve from 0.99286
945/945 ━━━━━━━━━━━━━━━━━━━━ 8s 8ms/step - accuracy: 1.0000 - loss: 4.6442e-04 - val_accuracy: 0.9929 - val_loss: 0.0257 - learning_rate: 1.0000e-06
Epoch 39/40
945/945 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.9999 - loss: 7.0599e-04
Epoch 39: val_accuracy did not improve from 0.99286
945/945 ━━━━━━━━━━━━━━━━━━━━ 8s 8ms/step - accuracy: 0.9999 - loss: 7.0581e-04 - val_accuracy: 0.9927 - val_loss: 0.0259 - l

In [10]:
from tensorflow.keras.callbacks import ModelCheckpoint, ReduceLROnPlateau

checkpoint_80 = ModelCheckpoint(
    "best_model_80.h5",
    monitor='val_accuracy',
    save_best_only=True,
    mode='max',
    verbose=1
)

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.3,
    patience=3,
    min_lr=1e-6,
    verbose=1
)

In [11]:
X80, y80 = load_data(80)

model_80 = build_model(X80.shape[1:])

history_80 = model_80.fit(
    X80, y80,
    validation_split=0.2,
    epochs=40,   # increased epochs
    batch_size=32,
    callbacks=[checkpoint_80, reduce_lr]
)

100%|██████████| 37800/37800 [03:47<00:00, 166.26it/s]
/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/40


2026-04-10 18:27:47.489939: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-04-10 18:27:47.631758: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.


941/945 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.6592 - loss: 1.0318
Epoch 1: val_accuracy improved from -inf to 0.95331, saving model to best_model_80.h5


945/945 ━━━━━━━━━━━━━━━━━━━━ 20s 12ms/step - accuracy: 0.6601 - loss: 1.0292 - val_accuracy: 0.9533 - val_loss: 0.1553 - learning_rate: 3.0000e-04
Epoch 2/40
943/945 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.9588 - loss: 0.1355
Epoch 2: val_accuracy improved from 0.95331 to 0.96772, saving model to best_model_80.h5


...

945/945 ━━━━━━━━━━━━━━━━━━━━ 9s 9ms/step - accuracy: 0.9868 - loss: 0.0414 - val_accuracy: 0.9839 - val_loss: 0.0478 - learning_rate: 3.0000e-04
Epoch 7/40
943/945 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.9881 - loss: 0.0359
Epoch 7: val_accuracy improved from 0.98386 to 0.98823, saving model to best_model_80.h5


945/945 ━━━━━━━━━━━━━━━━━━━━ 9s 9ms/step - accuracy: 0.9881 - loss: 0.0359 - val_accuracy: 0.9882 - val_loss: 0.0361 - learning_rate: 3.0000e-04
Epoch 8/40
941/945 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.9909 - loss: 0.0257
Epoch 8: val_accuracy did not improve from 0.98823
945/945 ━━━━━━━━━━━━━━━━━━━━ 9s 9ms/step - accuracy: 0.9909 - loss: 0.0257 - val_accuracy: 0.9593 - val_loss: 0.1110 - learning_rate: 3.0000e-04
Epoch 9/40
944/945 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.9907 - loss: 0.0292
Epoch 9: val_accuracy did not improve from 0.98823
945/945 ━━━━━━━━━━━━━━━━━━━━ 9s 9ms/step - accuracy: 0.9907 - loss: 0.0292 - val_accuracy: 0.9460 - val_loss: 0.1578 - learning_rate: 3.0000e-04
Epoch 10/40
941/945 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.9922 - loss: 0.0219
Epoch 10: val_accuracy did not improve from 0.98823

Epoch 10: ReduceLROnPlateau reducing learning rate to 9.000000427477062e-05.
945/945 ━━━━━━━━━━━━━━━━━━━━ 9s 9ms/step - accuracy: 0.9922 - loss: 0.0219

945/945 ━━━━━━━━━━━━━━━━━━━━ 9s 9ms/step - accuracy: 0.9959 - loss: 0.0125 - val_accuracy: 0.9914 - val_loss: 0.0251 - learning_rate: 9.0000e-05
Epoch 12/40
944/945 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.9983 - loss: 0.0059
Epoch 12: val_accuracy improved from 0.99140 to 0.99272, saving model to best_model_80.h5


945/945 ━━━━━━━━━━━━━━━━━━━━ 9s 9ms/step - accuracy: 0.9983 - loss: 0.0059 - val_accuracy: 0.9927 - val_loss: 0.0262 - learning_rate: 9.0000e-05
Epoch 13/40
944/945 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.9987 - loss: 0.0049
Epoch 13: val_accuracy did not improve from 0.99272
945/945 ━━━━━━━━━━━━━━━━━━━━ 9s 9ms/step - accuracy: 0.9987 - loss: 0.0049 - val_accuracy: 0.9913 - val_loss: 0.0265 - learning_rate: 9.0000e-05
Epoch 14/40
944/945 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.9993 - loss: 0.0033
Epoch 14: val_accuracy did not improve from 0.99272

Epoch 14: ReduceLROnPlateau reducing learning rate to 2.700000040931627e-05.
945/945 ━━━━━━━━━━━━━━━━━━━━ 9s 9ms/step - accuracy: 0.9993 - loss: 0.0033 - val_accuracy: 0.9915 - val_loss: 0.0270 - learning_rate: 9.0000e-05
Epoch 15/40
944/945 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.9996 - loss: 0.0028
Epoch 15: val_accuracy did not improve from 0.99272
945/945 ━━━━━━━━━━━━━━━━━━━━ 9s 9ms/step - accuracy: 0.9996 - loss: 0.

945/945 ━━━━━━━━━━━━━━━━━━━━ 9s 9ms/step - accuracy: 0.9998 - loss: 0.0013 - val_accuracy: 0.9929 - val_loss: 0.0261 - learning_rate: 2.4300e-06
Epoch 22/40
945/945 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.9999 - loss: 9.5157e-04
Epoch 22: val_accuracy did not improve from 0.99286
945/945 ━━━━━━━━━━━━━━━━━━━━ 9s 9ms/step - accuracy: 0.9999 - loss: 9.5158e-04 - val_accuracy: 0.9926 - val_loss: 0.0260 - learning_rate: 2.4300e-06
Epoch 23/40
945/945 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.9996 - loss: 0.0013
Epoch 23: val_accuracy did not improve from 0.99286

Epoch 23: ReduceLROnPlateau reducing learning rate to 1e-06.
945/945 ━━━━━━━━━━━━━━━━━━━━ 9s 9ms/step - accuracy: 0.9996 - loss: 0.0013 - val_accuracy: 0.9925 - val_loss: 0.0262 - learning_rate: 2.4300e-06
Epoch 24/40
940/945 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 1.0000 - loss: 7.9917e-04
Epoch 24: val_accuracy did not improve from 0.99286
945/945 ━━━━━━━━━━━━━━━━━━━━ 9s 9ms/step - accuracy: 1.0000 - loss: 8.0015

In [12]:
from tensorflow.keras.callbacks import ModelCheckpoint, ReduceLROnPlateau

checkpoint_128 = ModelCheckpoint(
    "best_model_128.h5",
    monitor='val_accuracy',
    save_best_only=True,
    mode='max',
    verbose=1
)

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.3,
    patience=3,
    min_lr=1e-6,
    verbose=1
)

In [13]:
X128, y128 = load_data(128)

model_128 = build_model(X128.shape[1:])

history_128 = model_128.fit(
    X128, y128,
    validation_split=0.2,
    epochs=40,   # increased epochs
    batch_size=32,
    callbacks=[checkpoint_128, reduce_lr]
)

100%|██████████| 37800/37800 [04:37<00:00, 136.18it/s]
/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/40


2026-04-10 18:38:33.415374: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-04-10 18:38:33.571967: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-04-10 18:38:33.891781: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-04-10 18:38:34.039776: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-04-10 18:38:34.743079: E external/local_xla/xla/stream_

943/945 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.6444 - loss: 1.0518
Epoch 1: val_accuracy improved from -inf to 0.88108, saving model to best_model_128.h5


945/945 ━━━━━━━━━━━━━━━━━━━━ 26s 16ms/step - accuracy: 0.6449 - loss: 1.0504 - val_accuracy: 0.8811 - val_loss: 0.3438 - learning_rate: 3.0000e-04
Epoch 2/40
944/945 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.9415 - loss: 0.1894
Epoch 2: val_accuracy improved from 0.88108 to 0.90317, saving model to best_model_128.h5


...

945/945 ━━━━━━━━━━━━━━━━━━━━ 13s 13ms/step - accuracy: 0.9996 - loss: 0.0020 - val_accuracy: 0.9911 - val_loss: 0.0338 - learning_rate: 2.7000e-05
Epoch 20/40
943/945 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.9998 - loss: 0.0013
Epoch 20: val_accuracy improved from 0.99114 to 0.99127, saving model to best_model_128.h5


945/945 ━━━━━━━━━━━━━━━━━━━━ 13s 13ms/step - accuracy: 0.9998 - loss: 0.0013 - val_accuracy: 0.9913 - val_loss: 0.0339 - learning_rate: 2.7000e-05
Epoch 21/40
941/945 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.9997 - loss: 0.0012
Epoch 21: val_accuracy did not improve from 0.99127

Epoch 21: ReduceLROnPlateau reducing learning rate to 8.100000013655517e-06.
945/945 ━━━━━━━━━━━━━━━━━━━━ 12s 13ms/step - accuracy: 0.9997 - loss: 0.0012 - val_accuracy: 0.9911 - val_loss: 0.0353 - learning_rate: 2.7000e-05
Epoch 22/40
941/945 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.9996 - loss: 0.0016
Epoch 22: val_accuracy improved from 0.99127 to 0.99153, saving model to best_model_128.h5


945/945 ━━━━━━━━━━━━━━━━━━━━ 13s 14ms/step - accuracy: 0.9996 - loss: 0.0016 - val_accuracy: 0.9915 - val_loss: 0.0318 - learning_rate: 8.1000e-06
Epoch 23/40
944/945 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.9999 - loss: 0.0013
Epoch 23: val_accuracy improved from 0.99153 to 0.99180, saving model to best_model_128.h5


945/945 ━━━━━━━━━━━━━━━━━━━━ 13s 13ms/step - accuracy: 0.9999 - loss: 0.0013 - val_accuracy: 0.9918 - val_loss: 0.0331 - learning_rate: 8.1000e-06
Epoch 24/40
945/945 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.9999 - loss: 8.7617e-04
Epoch 24: val_accuracy improved from 0.99180 to 0.99220, saving model to best_model_128.h5


945/945 ━━━━━━━━━━━━━━━━━━━━ 13s 13ms/step - accuracy: 0.9999 - loss: 8.7623e-04 - val_accuracy: 0.9922 - val_loss: 0.0331 - learning_rate: 8.1000e-06
Epoch 25/40
944/945 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.9997 - loss: 0.0011
Epoch 25: val_accuracy did not improve from 0.99220

Epoch 25: ReduceLROnPlateau reducing learning rate to 2.429999949526973e-06.
945/945 ━━━━━━━━━━━━━━━━━━━━ 13s 13ms/step - accuracy: 0.9997 - loss: 0.0011 - val_accuracy: 0.9913 - val_loss: 0.0350 - learning_rate: 8.1000e-06
Epoch 26/40
944/945 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.9999 - loss: 8.7055e-04
Epoch 26: val_accuracy did not improve from 0.99220
945/945 ━━━━━━━━━━━━━━━━━━━━ 12s 13ms/step - accuracy: 0.9999 - loss: 8.7061e-04 - val_accuracy: 0.9915 - val_loss: 0.0337 - learning_rate: 2.4300e-06
Epoch 27/40
942/945 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.9999 - loss: 8.9836e-04
Epoch 27: val_accuracy did not improve from 0.99220
945/945 ━━━━━━━━━━━━━━━━━━━━ 13s 13ms/step - 

In [14]:
def load_test_all():
    test_files = sorted(os.listdir(TEST_AUDIO_PATH))
    
    X64, X80, X128 = [], [], []
    
    for file in tqdm(test_files):
        file_path = f"{TEST_AUDIO_PATH}/{file}"
        
        mel64 = extract_features(file_path, 64)
        mel80 = extract_features(file_path, 80)
        mel128 = extract_features(file_path, 128)
        
        X64.append(mel64)
        X80.append(mel80)
        X128.append(mel128)
    
    X64 = np.array(X64)[..., np.newaxis]
    X80 = np.array(X80)[..., np.newaxis]
    X128 = np.array(X128)[..., np.newaxis]
    
    return X64, X80, X128, test_files

In [15]:
def tta_predict(model, X, n=7):
    preds = []
    
    for i in range(n):
        noise = np.random.normal(0, 0.01, X.shape)
        X_aug = X + noise
        preds.append(model.predict(X_aug))
    
    return np.mean(preds, axis=0)

In [16]:
X_test_64, X_test_80, X_test_128, test_files = load_test_all()

pred_64 = tta_predict(model_64, X_test_64)
pred_80 = tta_predict(model_80, X_test_80)
pred_128 = tta_predict(model_128, X_test_128)

100%|██████████| 16200/16200 [06:51<00:00, 39.39it/s]


507/507 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step
507/507 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step
507/507 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step
507/507 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step
507/507 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step
507/507 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step
507/507 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step
507/507 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step
507/507 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step
507/507 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step
507/507 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step
507/507 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step
507/507 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step
507/507 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step
507/507 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step
507/507 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step
507/507 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step
507/507 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step
507/507 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step
507/507 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step
507/507 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step


In [17]:
w1 = 0.3   # weight for model_64
w2 = 0.4   # weight for model_80
w3 = 0.3   # weight for model_128 (usually best → higher weight)

final_pred = (w1 * pred_64 + w2 * pred_80 + w3 * pred_128)

final_labels = np.argmax(final_pred, axis=1)

In [18]:
submission = pd.DataFrame({
    "id": [f.split('.')[0] for f in test_files],
    "digit": final_labels
})

submission.to_csv("submissi.csv", index=False)

print("Submission file created!")

Submission file created!
